# Clustering des données immobilières californiennes
## Bloc IA — Prosit 1

Ce notebook implémente le pipeline de clustering décrit dans le rapport :
1. Chargement et nettoyage des données
2. Sélection et standardisation des variables
3. Méthode du coude (Elbow Method) — WCSS
4. KMeans (k=6)
5. Visualisation et export du dataset transformé

## 1. Chargement et exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.rcParams['figure.figsize'] = (12, 8)
sns.set()
sns.set_context('talk')

In [ ]:
# Chargement du dataset original
housing = pd.read_csv('./data/housing.csv')
print(f"Shape : {housing.shape}")
housing.head()

In [ ]:
housing.info()

In [ ]:
# Valeurs manquantes
housing.isnull().sum()

## 2. Règles de nettoyage et préparation du dataset

D'après le rapport, le pipeline de transformation suit ces règles :

1. **Sélection des variables** : on retient uniquement `latitude`, `longitude`, `median_income` et `median_house_value`. Ces 4 variables captent la position géographique et le profil socio-économique.
2. **Gestion des valeurs manquantes** : seule `total_bedrooms` a 207 valeurs nulles, mais elle n'est pas utilisée pour le clustering. Les 4 variables sélectionnées n'ont aucune valeur manquante → aucune suppression de ligne nécessaire.
3. **Standardisation** : `StandardScaler` (centrage-réduction) pour que chaque variable ait la même influence sur la distance euclidienne.
4. **Pas de suppression d'outliers** : le rapport ne mentionne aucun filtre sur les valeurs extrêmes.

In [ ]:
# Sélection des 4 variables pour le clustering
features = ['latitude', 'longitude', 'median_income', 'median_house_value']
X = housing[features].copy()

# Vérification : aucune valeur manquante sur ces 4 colonnes
print("Valeurs manquantes sur les variables de clustering :")
print(X.isnull().sum())
print(f"\nLignes avant nettoyage : {len(X)}")

# Suppression des éventuelles lignes avec NaN (0 dans notre cas)
X = X.dropna()
print(f"Lignes après nettoyage : {len(X)}")

In [ ]:
# Standardisation (centrage-réduction)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Moyennes après standardisation :", X_scaled.mean(axis=0).round(2))
print("Écarts-types après standardisation :", X_scaled.std(axis=0).round(2))

## 3. Méthode du coude (Elbow Method) — WCSS

In [ ]:
# Calcul du WCSS pour k de 2 à 15
k_range = range(2, 16)
wcss = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    print(f"k={k:2d}  WCSS={kmeans.inertia_:.0f}")

In [ ]:
# Tracé de la courbe du coude
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), wcss, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=6, color='r', linestyle='--', label='k = 6 (coude)')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.title('Méthode du coude — WCSS en fonction de k')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Le coude est observé à **k = 6** : au-delà, la réduction du WCSS ralentit significativement.

## 4. KMeans avec k = 6

In [ ]:
# KMeans final
kmeans_final = KMeans(n_clusters=6, n_init=10, random_state=42)
clusters = kmeans_final.fit_predict(X_scaled)

# Ajout de la colonne cluster au dataframe original
housing['cluster'] = clusters

print("Répartition des clusters :")
print(housing['cluster'].value_counts().sort_index())

In [ ]:
# Statistiques par cluster
cluster_stats = housing.groupby('cluster').agg(
    effectif=('cluster', 'size'),
    prix_moyen=('median_house_value', 'mean'),
    revenu_moyen=('median_income', 'mean'),
    lat_moyenne=('latitude', 'mean'),
    lon_moyenne=('longitude', 'mean')
).round(2)

cluster_stats

## 5. Visualisation géographique des clusters

In [ ]:
# Scatter plot latitude vs longitude coloré par cluster
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    housing['longitude'], housing['latitude'],
    c=housing['cluster'], cmap='tab10',
    alpha=0.5, s=10
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Répartition spatiale des clusters (k=6)')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot revenu vs prix coloré par cluster
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    housing['median_income'], housing['median_house_value'],
    c=housing['cluster'], cmap='tab10',
    alpha=0.5, s=10
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Revenu médian')
plt.ylabel('Prix médian du logement ($)')
plt.title('Revenu vs Prix — coloré par cluster')
plt.tight_layout()
plt.show()

## 6. Export du dataset transformé

In [ ]:
# Export avec la colonne cluster ajoutée
output_path = './data/transformed_housing_data.csv'
housing.to_csv(output_path, index=False)
print(f"Dataset exporté : {output_path}")
print(f"Shape : {housing.shape}")
housing.head()

## Résumé du pipeline

| Étape | Description |
|-------|-------------|
| 1 | Chargement de `housing.csv` (20 640 lignes, 10 colonnes) |
| 2 | Suppression des lignes avec NaN sur les 4 variables de clustering (0 supprimées) |
| 3 | Standardisation avec `StandardScaler` |
| 4 | KMeans (k=6) → ajout de la colonne `cluster` |
| 5 | Export → `transformed_housing_data.csv` (20 640 lignes, 11 colonnes) |